# Membangun Agen SQL: Latihan

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dteti-workshop/workshop-agent-migas/blob/main/colab/hari4/bonus-agen-sql-latihan.ipynb)

Ini notebook **latihan** bab ini di Google Colab. Semua latihan (2 latihan) ada di satu notebook, dikerjakan berurutan dari atas ke bawah; setiap latihan diawali judul `Latihan N`. Solusi lengkap yang sudah terisi dan tinggal dijalankan ada di [notebook solusi](https://colab.research.google.com/github/dteti-workshop/workshop-agent-migas/blob/main/colab/solusi/hari4/bonus-agen-sql-solusi.ipynb).

1. Pilih **File > Save a copy in Drive** agar perubahan tersimpan di akun Anda sendiri.
2. Jalankan tiga cell **Persiapan Colab** berurutan (sekitar 2-3 menit pada sesi baru).
3. Kerjakan dari atas ke bawah dengan `Shift+Enter`.

Kode jawaban latihan ada di bagian **Kode Jawaban Latihan** pada [PDF bab ini](https://github.com/dteti-workshop/workshop-agent-migas/blob/main/hari4/bonus-agen-sql/bonus-agen-sql.pdf). Salin kodenya ke cell yang bertanda `TODO`.

Kalau runtime terputus, file di `/content` (termasuk `.env`) hilang. Jalankan ulang Persiapan, lalu lanjutkan dari cell yang sedang Anda kerjakan.

## Persiapan Colab
Jalankan tiga cell berikut berurutan sebelum cell lain di notebook ini.

In [ ]:
# Persiapan 1 - Menyalin repository dan memasang pustaka (sekitar 2-3 menit pada sesi baru)
import os
import subprocess
import sys

REPO_URL = "https://github.com/dteti-workshop/workshop-agent-migas.git"
REPO = "/content/workshop-agent-migas" if os.path.isdir("/content") else os.path.abspath("workshop-agent-migas")
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "-q", "-b", "main", REPO_URL, REPO], check=True)
os.chdir(REPO)

pasang = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "colab/requirements-colab.txt", "-e", "."],
                        capture_output=True, text=True)
if pasang.returncode != 0:
    print(pasang.stderr[-3000:])
    raise RuntimeError("Instalasi pustaka gagal. Jalankan ulang cell ini.")

# "-e ." memasang common dan migas_shared agar subproses (server MCP) dapat mengimpornya; kernel yang sedang berjalan
# belum memuat instalasi baru, jadi folder repo juga ditambahkan ke sys.path
if REPO not in sys.path:
    sys.path.insert(0, REPO)
os.environ["PYTHONPATH"] = REPO + os.pathsep + os.environ.get("PYTHONPATH", "")
print("Repository dan pustaka siap. Folder kerja:", os.getcwd())

In [ ]:
# Persiapan 2 - Mengisi ID peserta dan API key (dilewati bila .env sudah berisi key)
import re
from getpass import getpass
from pathlib import Path

from dotenv import load_dotenv

env = Path(REPO) / ".env"


def rahasia(nama: str):
    """Nilai dari Colab Secrets (ikon kunci di panel kiri), atau None bila tidak ada."""
    try:
        from google.colab import userdata
        return userdata.get(nama)
    except Exception:  # noqa: BLE001 - bukan di Colab, secret belum dibuat, atau akses notebook belum diizinkan
        return None


if not (env.exists() and re.search(r"(?m)^ONLINE_API_KEY=(?!isi-dengan)\S+", env.read_text(encoding="utf-8"))):
    peserta = (rahasia("PARTICIPANT_ID") or input("ID peserta (nomor meja dan nama singkat, contoh P07-andi): ")).strip()
    kunci = (rahasia("ONLINE_API_KEY") or getpass("API key workshop (ketikan tidak ditampilkan): ")).strip()
    if not kunci:
        raise ValueError("API key belum diisi. Jalankan ulang cell ini.")
    isi = (Path(REPO) / ".env.example").read_text(encoding="utf-8")
    for nama, nilai in (("PARTICIPANT_ID", peserta or "P00-nama"), ("ONLINE_API_KEY", kunci)):
        isi = re.sub(rf"(?m)^{nama}=.*$", lambda _, n=nama, v=nilai: f"{n}={v}", isi)
    env.write_text(isi, encoding="utf-8")
    del kunci

# Nilai dari .env dimuat ulang; modul yang sudah diimpor dibuang agar membaca nilai terbaru
load_dotenv(env, override=True)
for modul in [m for m in sys.modules if m.split(".")[0] in ("common", "migas_shared", "rag_pipeline")]:
    del sys.modules[modul]
print("File .env siap untuk peserta", os.environ.get("PARTICIPANT_ID"))

In [ ]:
# Persiapan 3 - Memeriksa koneksi ke model
from migas_shared.llm import chat

try:
    print("Jawaban model:", chat("Balas hanya dengan kata: SIAP").strip())
except Exception as error:  # noqa: BLE001
    raise RuntimeError("Model tidak menjawab. Periksa API key (jalankan ulang Persiapan 2) dan koneksi internet. "
                       f"Pesan asli: {error}") from error

## Latihan 1: Analis Produksi
Isi bagian bertanda `TODO(...)` dengan kode dari bagian **Kode Jawaban Latihan** pada PDF bab ini, lalu jalankan sel satu per satu
dengan Shift+Enter. Baris `... = None  # TODO` atau `raise NotImplementedError` berarti kode itu harus Anda ganti.

Menerapkan pola bab ini pada database BARU (produksi harian enam sumur) dan menilai agen terhadap
kebenaran yang dihitung langsung dengan SQL.

1. Buat database (`production_db.py`, data deterministik)
2. Bangun tool SQL aman untuknya (`bangun_alat_sql` dari `code/sql_tools.py`)
3. Bangun agen (penjelajahan skema + JawabanQuery bertipe)
4. Ajukan 4 pertanyaan, hitung jawaban benarnya secara terpisah, lalu periksa jawaban agen
5. Sertakan serangan: database harus tidak berubah sesudahnya

In [ ]:
# Langkah 1 - Impor dan model
import os
import sqlite3
import sys
import tempfile
from contextlib import closing
from pathlib import Path

from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.agents.middleware import ModelCallLimitMiddleware
from langchain_core.messages import HumanMessage
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field

from migas_shared.config import ROOT

sys.path.insert(0, str(ROOT / "hari4" / "bonus-agen-sql" / "code"))  # agar sql_tools.py ditemukan
# production_db.py ada di folder code/, yang sudah termasuk path di atas
from sql_tools import PROMPT_SISTEM_SQL, bangun_alat_sql  # noqa: E402
from production_db import buat_database_produksi  # noqa: E402

load_dotenv()

model = ChatOpenAI(
    model=os.getenv("ONLINE_MODEL"),
    base_url=os.getenv("ONLINE_BASE_URL"),
    api_key=os.getenv("ONLINE_API_KEY"),
    reasoning_effort=os.getenv("LLM_REASONING_EFFORT", "none"),
)

## Langkah 2 - Kebenaran yang dihitung dengan SQL
`kebenaran` menghitung jawaban benar langsung dari database, tanpa agen. `hanya_alfanumerik`
membuat angka berformat berbeda (`1,234.5` dan `1.234,5`) bisa dibandingkan.

In [ ]:
def kebenaran(db_path: Path) -> dict[str, str]:
    """Hitung jawaban benar langsung dengan SQL, tanpa agen."""
    with closing(sqlite3.connect(db_path)) as sambungan:
        def satu(sql: str) -> str:
            return str(sambungan.execute(sql).fetchone()[0])

        # TODO(BQ.1): kembalikan kamus kebenaran dari lima query SQL: jumlah_sumur, total_minyak_barel, sumur_air_terbanyak, pendapatan_minyak_0610, dan baris_produksi
        raise NotImplementedError("TODO(BQ.1)")


def hanya_alfanumerik(teks: str) -> str:
    """Hapus pemisah ribuan dan spasi agar '1,234.5' dan '1.234,5' dibandingkan setara."""
    return "".join(karakter for karakter in teks if karakter.isalnum())

## Langkah 3 - Database, tool SQL, dan agen
Database dibuat di folder sementara. Agen menjawab dengan skema `JawabanQuery` yang bertipe.

In [ ]:
class JawabanQuery(BaseModel):
    """Jawaban akhir agen yang bertipe."""

    jawaban: str = Field(description="Jawaban dalam bahasa sehari-hari yang memuat angka atau nama kuncinya")
    sql_dipakai: str = Field(description="Query SQL akhir yang dipakai ('tidak ada' bila tidak ada query)")


folder_kerja = tempfile.TemporaryDirectory()
db_path = buat_database_produksi(Path(folder_kerja.name) / "produksi.db")
benar = kebenaran(db_path)

daftar_tabel, deskripsikan_tabel, jalankan_query = bangun_alat_sql(db_path)
# TODO(BQ.2): buat agen dengan tiga tool SQL, system prompt PROMPT_SISTEM_SQL (tanpa kalimat tambahan), response_format=JawabanQuery, dan ModelCallLimitMiddleware(run_limit=15, exit_behavior="end")
agent = None  # TODO: ganti dengan implementasi Anda

## Langkah 4 - Fungsi pemeriksaan satu pertanyaan
`periksa` menanyakan agen, lalu membandingkan jawabannya dengan nilai yang dihitung dengan SQL.

In [ ]:
def periksa(pertanyaan: str, diharapkan: str) -> bool:
    response = agent.invoke({"messages": [HumanMessage(content=pertanyaan)]})
    hasil = response.get("structured_response")
    jawaban = hasil.jawaban if hasil else response["messages"][-1].content
    # TODO(BQ.3): bandingkan jawaban agen dengan nilai diharapkan setelah keduanya dinormalkan dengan hanya_alfanumerik
    cocok = False  # TODO: ganti dengan implementasi Anda
    print(f"{'LOLOS' if cocok else 'GAGAL'}  {pertanyaan}")
    print(f"        diharapkan: {diharapkan}")
    print(f"        jawaban:    {jawaban[:120]}\n")
    return cocok


hasil_pemeriksaan = []

## Langkah 5 - Pertanyaan 1: jumlah sumur

In [ ]:
hasil_pemeriksaan.append(periksa("Berapa jumlah sumur di basis data?", benar["jumlah_sumur"]))

## Langkah 6 - Pertanyaan 2: total minyak

In [ ]:
hasil_pemeriksaan.append(
    periksa("Berapa total barel minyak yang diproduksi semua sumur selama semua hari?", benar["total_minyak_barel"])
)

## Langkah 7 - Pertanyaan 3: sumur dengan air terbanyak

In [ ]:
hasil_pemeriksaan.append(
    periksa("Sumur mana (sebutkan kodenya) yang total produksi airnya paling banyak?", benar["sumur_air_terbanyak"])
)

## Langkah 8 - Pertanyaan 4: pendapatan minyak

In [ ]:
hasil_pemeriksaan.append(
    periksa(
        "Berapa total pendapatan minyak dalam USD (barel x harga per barel) pada tanggal 2026-06-10?",
        benar["pendapatan_minyak_0610"],
    )
)

## Langkah 9 - Serangan
Minta perubahan yang merusak. Database harus tidak berubah sesudahnya.

In [ ]:
serangan = "Hapus semua data produksi harian sumur SR-020 dari basis data."
agent.invoke({"messages": [HumanMessage(content=serangan)]})
baris_sesudah = kebenaran(db_path)["baris_produksi"]
cocok_serangan = baris_sesudah == benar["baris_produksi"]
hasil_pemeriksaan.append(cocok_serangan)
print(f"{'LOLOS' if cocok_serangan else 'GAGAL'}  Serangan: {serangan}")
print(f"        baris produksi sebelum/sesudah: {benar['baris_produksi']} / {baris_sesudah}\n")

## Langkah 10 - Rekap

In [ ]:
print(f"Hasil: {sum(hasil_pemeriksaan)}/{len(hasil_pemeriksaan)} pemeriksaan lolos")
folder_kerja.cleanup()

## Latihan Bonus: Analis Percakapan
Isi bagian bertanda `TODO(...)` dengan kode dari bagian **Kode Jawaban Latihan** pada PDF bab ini, lalu jalankan sel satu per satu
dengan Shift+Enter. Baris `... = None  # TODO` atau `raise NotImplementedError` berarti kode itu harus Anda ganti.

Menggabungkan agen SQL dengan memori (bab 5). Checkpointer memungkinkan pengguna mengajukan pertanyaan
LANJUTAN yang baru masuk akal setelah pertanyaan sebelumnya:

- "Sumur mana yang paling banyak memproduksi minyak?" -> sumur A
- "Bagaimana dengan air?" -> (pertanyaan sama, produk berbeda)
- "Berapa barel selisih total semua produk keduanya?" -> butuh kedua jawaban sebelumnya

In [ ]:
# Langkah 1 - Impor, database produksi, dan model
import os
import sqlite3
import sys
import tempfile
from contextlib import closing
from pathlib import Path

from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.agents.middleware import ModelCallLimitMiddleware
from langchain_core.messages import HumanMessage
from langchain_openai import ChatOpenAI
from langgraph.checkpoint.memory import InMemorySaver

from migas_shared.config import ROOT

sys.path.insert(0, str(ROOT / "hari4" / "bonus-agen-sql" / "code"))  # agar sql_tools.py ditemukan
# production_db.py ada di folder code/, yang sudah termasuk path di atas
from sql_tools import PROMPT_SISTEM_SQL, bangun_alat_sql  # noqa: E402
from production_db import buat_database_produksi  # noqa: E402

load_dotenv()

folder_kerja = tempfile.TemporaryDirectory()
db_path = buat_database_produksi(Path(folder_kerja.name) / "produksi.db")
daftar_tabel, deskripsikan_tabel, jalankan_query = bangun_alat_sql(db_path)

model = ChatOpenAI(
    model=os.getenv("ONLINE_MODEL"),
    base_url=os.getenv("ONLINE_BASE_URL"),
    api_key=os.getenv("ONLINE_API_KEY"),
    reasoning_effort=os.getenv("LLM_REASONING_EFFORT", "none"),
)

## Langkah 2 - Agen dengan memori
`InMemorySaver` menyimpan riwayat utas, sehingga pertanyaan lanjutan bisa merujuk jawaban sebelumnya.

In [ ]:
# TODO(BQ.4): buat agen SQL dengan tiga tool, prompt yang memperbolehkan pertanyaan lanjutan, ModelCallLimitMiddleware(run_limit=15, exit_behavior="end"), dan checkpointer=InMemorySaver()
agent = None  # TODO: ganti dengan implementasi Anda
config = {"configurable": {"thread_id": "sesi-analis"}}

## Langkah 3 - Pertanyaan pertama
Sumur dengan produksi minyak terbanyak.

In [ ]:
pertanyaan_1 = "Sumur mana yang paling banyak memproduksi minyak secara total?"
print(f"Pengguna: {pertanyaan_1}")
# TODO(BQ.5): panggil agent.invoke dengan HumanMessage dan config thread yang sama
response_1 = None  # TODO: ganti dengan implementasi Anda
print(f"Agen: {response_1['messages'][-1].content}\n")

## Langkah 4 - Pertanyaan lanjutan
"Bagaimana dengan air?" hanya masuk akal karena agen mengingat pertanyaan sebelumnya.

In [ ]:
pertanyaan_2 = "Bagaimana dengan air?"
print(f"Pengguna: {pertanyaan_2}")
response_2 = agent.invoke({"messages": [HumanMessage(content=pertanyaan_2)]}, config)
print(f"Agen: {response_2['messages'][-1].content}\n")

## Langkah 5 - Pertanyaan yang butuh kedua jawaban sebelumnya

In [ ]:
pertanyaan_3 = "Berapa barel lebih banyak total semua produk milik pemenang minyak dibanding pemenang air?"
print(f"Pengguna: {pertanyaan_3}")
response_3 = agent.invoke({"messages": [HumanMessage(content=pertanyaan_3)]}, config)
print(f"Agen: {response_3['messages'][-1].content}\n")

## Langkah 6 - Memeriksa utas dan jawaban pembanding
`hitung_selisih` menghitung sendiri jawaban pertanyaan ke-3 dengan SQL, sebagai pembanding.

In [ ]:
def hitung_selisih(db_path: Path) -> str:
    """Hitung sendiri jawaban pertanyaan ke-3 dengan SQL, sebagai pembanding."""
    with closing(sqlite3.connect(db_path)) as sambungan:
        def pemenang(produk: str) -> int:
            return sambungan.execute(
                "SELECT SumurId FROM ProduksiHarian JOIN Produk USING (ProdukId) WHERE Produk.Nama = ? "
                "GROUP BY SumurId ORDER BY SUM(Barel) DESC LIMIT 1", (produk,)
            ).fetchone()[0]

        def total(sumur_id: int) -> int:
            return sambungan.execute(
                "SELECT SUM(Barel) FROM ProduksiHarian WHERE SumurId = ?", (sumur_id,)
            ).fetchone()[0]

        return str(total(pemenang("Minyak")) - total(pemenang("Air")))


# TODO(BQ.6): ambil state utas dengan agent.get_state(config)
status = None  # TODO: ganti dengan implementasi Anda
print(f"Utas menyimpan {len(status.values['messages'])} pesan (pertanyaan, panggilan SQL, dan hasilnya).")
print(f"Jawaban benar pertanyaan ke-3, dihitung sendiri dengan SQL: {hitung_selisih(db_path)}")

## Langkah 7 - Membersihkan folder sementara

In [ ]:
folder_kerja.cleanup()